## 延伸 gemma4_tool_calling_1 的實作

* **新增 Web Search Tool**
* **修改 System Prompt**：給予其回應時應遵循的原則性命令描述，特定的工具使用方法則定義在 Tool Schema 當中。
* **調整 Tool Schema 工具使用**：避免在 `gemma4_tool_calling_1` 中模型錯誤理解使用工具的情況。例如在之前的版本中，模型會把查詢「板橋」的天氣命令直接理解成「板橋，Hwanghae-bukto，朝鲜」，但實際上希望查詢的地方為「台灣新北市板橋」。

---

### Web Search 工具使用 tavily

* 新增 `WEB_SEARCH_TOOL` Schema
* 將 `WEB_SEARCH_TOOL` 新增至 `TOOLS` 列表
* 定義 `web_search` 函式
* `TOOL_REGISTRY` 加入 `web_search` 鍵值

In [1]:
!pip install -q tavily-python

In [2]:
import torch
import transformers
import pydantic

print('PyTorch', torch.__version__)
print('Transformers', transformers.__version__)
print('Pydantic', pydantic.__version__)

assert torch.cuda.is_available(), '目前沒偵測到 GPU，請切換到運行 GPU 的執行模式'
print('GPU', torch.cuda.get_device_name(0))

PyTorch 2.11.0+cu130
Transformers 5.17.0
Pydantic 2.13.5
GPU NVIDIA A100-SXM4-80GB


In [3]:
from huggingface_hub import login

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("已使用 Colab Secret 登入 Hugging Face。")
else:
    print(
        "未設定 HF_TOKEN。若模型可以公開下載，仍可繼續；"
        "若出現 401 或 403，請設定 Hugging Face Token。"
    )

已使用 Colab Secret 登入 Hugging Face。


In [4]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    padding_side="left",
)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
    attn_implementation="sdpa",
)

model.eval()

print("模型載入完成：", MODEL_ID)
print("模型主要裝置：", model.device)

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 10.2GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

模型載入完成： google/gemma-4-E2B-it
模型主要裝置： cuda:0


In [5]:
for name, token in processor.tokenizer.special_tokens_map.items():
    token_id = processor.tokenizer.convert_tokens_to_ids(token)
    print(f'{name} -> {token} -> {token_id}')

bos_token -> <bos> -> 2
eos_token -> <eos> -> 1
unk_token -> <unk> -> 3
pad_token -> <pad> -> 0
mask_token -> <mask> -> 4
audio_token -> <|audio|> -> 258881
boa_token -> <|audio> -> 256000
boi_token -> <|image> -> 255999
eoa_token -> <audio|> -> 258883
eoc_token -> <channel|> -> 101
eoi_token -> <image|> -> 258882
eot_token -> <turn|> -> 106
escape_token -> <|"|> -> 52
etc_token -> <tool_call|> -> 49
etd_token -> <tool|> -> 47
etr_token -> <tool_response|> -> 51
image_token -> <|image|> -> 258880
soc_token -> <|channel> -> 100
sot_token -> <|turn> -> 105
stc_token -> <|tool_call> -> 48
std_token -> <|tool> -> 46
str_token -> <|tool_response> -> 50
think_token -> <|think|> -> 98


In [6]:
from typing import Literal, Any
from pydantic import BaseModel, Field, field_validator


MAX_TOOL_ROUNDS = 3

# 先關閉 thinking，速度較快。
# 若模型經常沒有正確選擇工具，可以改成 True。
ENABLE_THINKING = False


SYSTEM_PROMPT = """
你是一個繁體中文 AI 助理。
規則：
1. 可以由一般穩定知識回答的問題，直接回答。
2. 需要即時、近期或外部資訊時，使用適合的工具。
3. 如果有專門處理該資訊的工具，優先使用專用工具，不要使用較泛用的工具替代。
4. 不確定必要的工具參數時，先詢問使用者，不要自行猜測。
5. 工具回傳失敗時，不得捏造資料。
6. 最終回答使用繁體中文。
7. 面對具備多義性、地域差異或缺乏明確上下文的提問時，不得自行預設單一情境或立場。應客觀列出可能的多種情況，並主動引導使用者釐清確切意圖。
""".strip()

# Tool Schema
WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_current_weather",

        "description": (
            "查詢指定地點目前的即時天氣。"
            "適用於目前溫度、體感溫度、濕度、降雨、"
            "雲量與風速等即時天氣資訊。"
            "如果使用者提供的是非英文地名，"
            "應將地名轉換成常用英文或羅馬拼音後再呼叫此工具。"
            "必須保留使用者真正指定的地點，"
            "不得自行替換成附近較大的城市。"
            "你需要對指定地點進行全世界可能所屬的國家判斷"
            "信心程度非常高可以明確推斷其所屬國家時，"
            "請自行推論並判斷該地的所屬國家，"
            "以代入工具的 country_code 參數當中。"
            "當信心程度有些微不確定無法肯定指定地點所屬的國家或第一行政區時，"
            "請先用自然語言回覆，"
            "表明不確定所詢問地點的所屬國家或其所屬行政區，"
            "並列出該地點所指的可能國家，"
            "讓使用者判斷回覆。"
        ),

        "parameters": {
            "type": "object",

            "properties": {

                "location": {
                    "type": "string",
                    "description": (
                        "要查詢的城市、區域或地點名稱。"
                        "必須使用常用英文名稱或羅馬拼音，"
                        "不要直接使用中文、日文、韓文等非英文地名。"
                        "只放實際地點名稱，不要把國家或 admin1 "
                        "合併在這個參數中。"
                        "例如："
                        "東京 -> 'Tokyo'；"
                        "首爾 -> 'Seoul'；"
                        "洛杉磯 -> 'Los Angeles'。"
                        "【重要規則】：如果使用者提供的地名可能存在歧義（例如在多個國家都有，如『Perth』），"
                        "且使用者並未提供國家資訊，請**不要**呼叫此工具！"
                        "你必須先用自然語言回覆，"
                        "告訴使用者有哪幾個國家都有同樣的地名，"
                        "詢問他們具體是指哪個國家或哪個一級行政區的地點。"
                    ),
                },

                "country_code": {
                    "type": "string",
                    "description": (
                        "使用者提供國家資訊時，"
                        "填入 ISO 3166-1 alpha-2 的兩碼大寫國家代碼。"
                        "例如："
                        "台灣 -> 'TW'；"
                        "日本 -> 'JP'；"
                        "韓國 -> 'KR'；"
                        "美國 -> 'US'。"
                        "如果使用者沒有提供國家，"
                        "且無法可靠判斷國家，可以省略此參數。"
                    ),
                },

                "admin1": {
                    "type": "string",
                    "description": (
                        "第一級行政區的英文名稱或正式縮寫。"
                        "只有在使用者提供第一級行政區資訊時填入。"
                        "例如："
                        "台北市 -> 'Taipei City'；"
                        "California -> 'California'。"
                        "此值將用來縮小 geocoding 搜尋結果。"
                        "如果沒有提供或無法可靠判斷，可以省略。"
                    ),
                },

                "unit": {
                    "type": "string",
                    "enum": [
                        "celsius",
                        "fahrenheit",
                    ],
                    "description": (
                        "溫度單位。"
                        "使用者未指定時使用 celsius。"
                    ),
                },
            },

            "required": [
                "location",
                "country_code"
            ],
        },
    },
}

WEB_SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "web_search",

        "description": (
            "搜尋網路上的外部或最新資訊。"
            "若使用者提到今天、最近、本週、"
            "本月等時間條件，應設定適當的 time_range。"
        ),

        "parameters": {
            "type": "object",

            "properties": {

                "query": {
                    "type": "string",
                    "description": (
                        "搜尋關鍵字。"
                    )
                },

                "max_results": {
                    "type": "integer",
                    "description": (
                        "最多搜尋幾筆結果。"
                    )
                },

                "time_range": {
                    "type": "string",
                    "enum": [
                        "day",
                        "week",
                        "month",
                        "year"
                    ],
                    "description": (
                        "搜尋結果的時間範圍。"
                        "day 表示最近一天，"
                        "week 表示最近一週，"
                        "month 表示最近一個月，"
                        "year 表示最近一年。"
                        "沒有時間限制時可省略。"
                    )
                },

                "topic": {
                    "type": "string",
                    "enum": [
                        "general",
                        "news"
                    ],
                    "description": (
                        "新聞事件使用 news；"
                        "一般網路搜尋使用 general。"
                    )
                }
            },

            "required": [
                "query"
            ]
        }
    }
}

TOOLS = [
    WEATHER_TOOL,
    WEB_SEARCH_TOOL
         ]


class WeatherArguments(BaseModel):
    """實際執行工具前，用來驗證模型產生的參數。"""

    location: str = Field(min_length=2,max_length=100)

    country_code: str | None = Field(default=None,min_length=2,max_length=2)

    admin1: str | None = Field(default=None,min_length=2,max_length=100)

    unit: Literal["celsius","fahrenheit"] = "celsius"


    @field_validator("country_code")
    @classmethod
    def normalize_country_code(cls,value: str | None) -> str | None:

        if value is None:
            return None

        value = value.strip().upper()

        if not value.isalpha():
            raise ValueError("country_code 必須是兩個英文字母。")

        return value


    @field_validator("location","admin1",mode="before")
    @classmethod
    def strip_text(cls,value):

        if isinstance(value, str):
            value = value.strip()

        return value

class WebSearchArguments(BaseModel):

    query: str = Field(
        min_length=2,
        max_length=400
    )

    max_results: int = Field(
        default=5,
        ge=1,
        le=10
    )

    time_range: Literal[
        "day",
        "week",
        "month",
        "year"
    ] | None = None

    topic: Literal[
        "general",
        "news"
    ] = "general"

In [7]:
import requests


WMO_WEATHER_CODES = {
    0: "晴朗",
    1: "大致晴朗",
    2: "局部多雲",
    3: "陰天",
    45: "有霧",
    48: "霧淞",
    51: "輕微毛毛雨",
    53: "中等毛毛雨",
    55: "強烈毛毛雨",
    56: "輕微凍毛毛雨",
    57: "強烈凍毛毛雨",
    61: "小雨",
    63: "中雨",
    65: "大雨",
    66: "輕微凍雨",
    67: "強烈凍雨",
    71: "小雪",
    73: "中雪",
    75: "大雪",
    77: "雪粒",
    80: "輕微陣雨",
    81: "中等陣雨",
    82: "強烈陣雨",
    85: "輕微陣雪",
    86: "強烈陣雪",
    95: "雷雨",
    96: "雷雨伴隨小冰雹",
    99: "雷雨伴隨強冰雹",
}

HTTP = requests.Session()

HTTP.headers.update({
    "User-Agent": "gemma4-colab-tool-calling-demo/1.0"
})

def geocode_location(location: str, country_code: str | None = None, admin1: str | None = None) -> dict[str, Any]:
    """
    將地點轉換成 Open-Meteo 可使用的經緯度。

    Args:
        location:
            英文或羅馬拼音地點名稱，
            例如 Banqiao、Tokyo。

        country_code:
            ISO 3166-1 alpha-2 國家代碼，
            例如 TW、JP。

        admin1:
            第一級行政區英文名稱，
            例如 New Taipei、California。

    Returns:
        Open-Meteo geocoding result。
    """

    location = location.strip()

    if not location:
        raise ValueError("location 不可為空。")

    # ---------------------------------
    # Open-Meteo 的 name 格式：
    #
    # <location>
    # 或
    # <location>, <admin1>
    # ---------------------------------

    if admin1:
        search_name = (f"{location}, {admin1.strip()}")
    else:
        search_name = location


    params = {
        "name": search_name,

        # 不只取 1 筆，方便後續做結果驗證
        "count": 10,

        # 因為 Gemma 已被要求產生英文地名，
        # geocoding 也統一使用英文
        "language": "en",

        "format": "json",
    }

    # ---------------------------------
    # countryCode 是 Open-Meteo
    # 獨立的國家 filter
    # ---------------------------------

    if country_code:
        params["countryCode"] = (country_code.strip().upper())

    response = HTTP.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params=params,
        timeout=20,
    )

    response.raise_for_status()
    payload = response.json()
    results = (payload.get("results") or [])


    # ---------------------------------
    # 第一輪：
    # location + admin1 + country
    # 找不到時，退回 location + country
    #
    # 避免 Gemma 的 admin1 英文名稱
    # 與 Open-Meteo 索引名稱略有差異
    # ---------------------------------

    if (not results and admin1):
        fallback_params = {
            "name": location,
            "count": 10,
            "language": "en",
            "format": "json",
        }

        if country_code:
            fallback_params["countryCode"] = (country_code.strip().upper())

        response = HTTP.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params=fallback_params,
            timeout=20,
        )

        response.raise_for_status()
        payload = response.json()
        results = (payload.get("results") or [])


    if not results:
        search_description = location

        if admin1:
            search_description += (f", {admin1}")

        if country_code:
            search_description += (f" ({country_code})")

        raise ValueError("找不到地點："f"{search_description}")

    # ---------------------------------
    # 由於 countryCode 已在 API 層 filter，
    # 第一筆原則上已位於指定國家。
    # ---------------------------------

    place = results[0]

    # 再做一次 defensive validation
    if country_code:

        returned_country_code = (
            place.get("country_code")
        )

        if (returned_country_code and returned_country_code.upper() != country_code.upper()):
            raise ValueError(
                "Geocoding 回傳國家與要求不符："
                f"expected={country_code}, "
                f"actual={returned_country_code}"
            )

    return {
        "name": place.get("name"),
        "admin1": place.get("admin1"),
        "admin2": place.get("admin2"),
        "country": place.get("country"),
        "country_code": place.get("country_code"),
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "timezone": place.get("timezone"),
    }


def get_current_weather(
    location: str,
    country_code: str | None = None,
    admin1: str | None = None,
    unit: str = "celsius",
) -> dict[str, Any]:

    """
    查詢指定地點目前的天氣。

    Args:
        location:
            英文或羅馬拼音地點名稱。

        country_code:
            ISO 3166-1 alpha-2 國家代碼。

        admin1:
            第一級行政區英文名稱。

        unit:
            celsius 或 fahrenheit。

    Returns:
        包含目前天氣資料的字典。
    """

    # =================================
    # Step 1
    # 地名 → 經緯度
    # =================================

    place = geocode_location(
        location=location,
        country_code=country_code,
        admin1=admin1,
    )


    # =================================
    # Step 2
    # 經緯度 → Current Weather
    # =================================

    response = HTTP.get(
        "https://api.open-meteo.com/v1/forecast",

        params={
            "latitude":
                place["latitude"],

            "longitude":
                place["longitude"],

            "current": (
                "temperature_2m,"
                "relative_humidity_2m,"
                "apparent_temperature,"
                "precipitation,"
                "weather_code,"
                "cloud_cover,"
                "wind_speed_10m"
            ),

            "temperature_unit": (
                "fahrenheit"
                if unit == "fahrenheit"
                else "celsius"
            ),

            "wind_speed_unit": "kmh",

            # 使用地點當地 timezone
            "timezone": "auto",
        },

        timeout=20,
    )

    response.raise_for_status()

    payload = response.json()

    current = payload.get(
        "current"
    )

    if not current:

        raise RuntimeError(
            "Open-Meteo 沒有回傳 "
            "current weather 資料。"
        )


    weather_code = current.get(
        "weather_code"
    )


    # =================================
    # 組合地點名稱
    # =================================

    location_parts = [
        place.get("name"),
        place.get("admin1"),
        place.get("country"),
    ]

    resolved_location = "，".join(
        str(part)
        for part in location_parts
        if part
    )


    # =================================
    # 回傳給 Gemma 的 tool response
    # =================================

    return {

        "requested_location":
            location,

        "requested_admin1":
            admin1,

        "requested_country_code":
            country_code,

        "resolved_location":
            resolved_location,

        "resolved_country_code":
            place.get("country_code"),

        "latitude":
            place["latitude"],

        "longitude":
            place["longitude"],

        "timezone":
            (
                payload.get("timezone")
                or place.get("timezone")
            ),

        "observation_time":
            current.get("time"),

        "temperature":
            current.get(
                "temperature_2m"
            ),

        "temperature_unit":
            (
                payload
                .get(
                    "current_units",
                    {},
                )
                .get(
                    "temperature_2m"
                )
            ),

        "apparent_temperature":
            current.get(
                "apparent_temperature"
            ),

        "relative_humidity_percent":
            current.get(
                "relative_humidity_2m"
            ),

        "precipitation_mm":
            current.get(
                "precipitation"
            ),

        "cloud_cover_percent":
            current.get(
                "cloud_cover"
            ),

        "wind_speed_kmh":
            current.get(
                "wind_speed_10m"
            ),

        "weather_code":
            weather_code,

        "weather_description":
            WMO_WEATHER_CODES.get(
                weather_code,
                "未知天氣狀況"
            ),

        "data_source":
            "Open-Meteo",
    }



In [8]:
from tavily import TavilyClient

try:
    from google.colab import userdata
    TAVILY_API_KEY = userdatas.get('TAVILY_API_KEY')
except Exception:
    TAVILY_API_KEY = None

tavily_client = TavilyClient(api_key = TAVILY_API_KEY)

def web_search(
    query: str,
    max_results: int = 5,
    time_range: str | None = None,
    topic: str = "general"
) -> dict:

    params = {
        "query": query,
        "search_depth": "basic",
        "max_results": max_results,
        "topic": topic,
        "include_answer": False,
        "include_raw_content": False,
    }

    if time_range is not None:
        params["time_range"] = time_range

    response = tavily_client.search(
        **params
    )

    results = []

    for item in response.get(
        "results",
        []
    ):

        results.append({
            "title": item.get("title"),
            "url": item.get("url"),
            "content": item.get("content"),
            "score": item.get("score"),
        })

    return {
        "query": query,
        "time_range": time_range,
        "topic": topic,
        "results": results,
    }

In [9]:
test_get_weather = get_current_weather('Taipei')

test_get_weather

{'requested_location': 'Taipei',
 'requested_admin1': None,
 'requested_country_code': None,
 'resolved_location': 'Taipei，Taiwan，Taiwan',
 'resolved_country_code': 'TW',
 'latitude': 25.05306,
 'longitude': 121.52639,
 'timezone': 'Asia/Taipei',
 'observation_time': '2026-10-03T12:45',
 'temperature': 28.9,
 'temperature_unit': '°C',
 'apparent_temperature': 32.7,
 'relative_humidity_percent': 70,
 'precipitation_mm': 0.1,
 'cloud_cover_percent': 70,
 'wind_speed_kmh': 15.9,
 'weather_code': 51,
 'weather_description': '輕微毛毛雨',
 'data_source': 'Open-Meteo'}

In [10]:
TOOL_REGISTRY = {
    'get_current_weather':{
        'function': get_current_weather,
        'validator': WeatherArguments
    },
    'web_search':{
        'function': web_search,
        'validator': WebSearchArguments
        }
}

In [11]:
import json
import re

def cast_tool_value(value:str) -> Any:
    """
    把 tool call 中的文字轉成適當的 Python 型別
    """
    value = value.strip()

    if value.lower() == 'true':
        return True
    if value.lower() == 'false':
        return False
    if value.lower() == 'null':
        return None

    try:
        return int(value)
    except ValueError:
        pass

    try:
        return float(value)
    except ValueError:
        pass

    return value.strip("'\"")


def extract_tool_call_with_regex(text:str) -> list[dict[str, Any]]:
    """
    解析 Gemma 4 原始的 tool-call token 格式
    """
    calls = []
    matches = re.findall(
        r'<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>',
        text,
        flags = re.DOTALL
    )

    for function_name, raw_args in matches:
        arguments = {}

        arguments_matches = re.findall(
            r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))',
            raw_args,
            flags = re.DOTALL
        )
        for key, quotated_value, plain_value in arguments_matches:
            raw_value = (
                quotated_value
                if quotated_value != ''
                else palin_value
            )

            arguments[key] = cast_tool_value(raw_value)

        calls.append(
            {
                'name':function_name,
                'arguments':arguments
            }
        )
    return calls

def normalize_parsed_tool_calls(parsed_calls: list[dict[str,Any]] | None) -> list[dict[str,Any]]:
    """
    將 processor.parse_response 的結果統一成:
    {
    'name':...,
    'arguments':{...}
    }
    """
    normalized = []

    for call in parsed_calls or []:
        function = call.get('function')

        name = function.get('name')
        arguments = function.get('arguments')

        if isinstance(arguments, str):
            try:
                arguments = json.loads(arguments)
            except JSONDecodeError:
                arguments = {}
        if name:
            normalized.append({
                'name': name,
                'arguments': arguments
            })
    return normalized


def clean_generated_text(text:str) -> str:
    """
    移除模型回答中的 special token
    """
    cleaned = text

    tokens_to_remove = (

        # 模型生成結束 token
        '<eos>',
        '<turn|>',

        # Function calling protocol
        '<|tool_call>',
        '<tool_call|>',
        '<|tool_response>',
        '<tool_response|>',

        # Thinking channel protocol
        '<|channel>',
        '<channel|>'

    )
    for token in tokens_to_remove:
        cleaned = cleaned.replace(token,'')

    cleaned = re.sub(
        r'call:\w\{.*?\}',
        '',
        cleaned
    )

    return cleaned.strip()

def parse_model_output(raw_output:str) -> dict[str, Any]:
    """
    將 Gemma 4 輸出解析成一般回答或 tool call
    """

    if hasattr(processor, 'parse_response'):
        try:
            parsed = processor.parse_response(raw_output)
            tool_calls = normalize_parsed_tool_calls(
                parsed.get('tool_calls')
            )

            content = (
                parsed.get('content') or ''
            ).strip()

            if tool_calls or content:
                return {
                    'content':content,
                    'tool_calls':tool_calls,
                    'thinking':parsed.get('thinking')
                }
        except Exception:
            pass

    tool_calls = extract_tool_call_with_regex(raw_output)

    return {
        'content':(
            ''
            if tool_calls
            else clean_generated_text(raw_output)
        ),
        'tool_calls': tool_calls,
        'thinking': None
    }

In [12]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": "新竹現在天氣如何？"
    }
]
prompt = processor.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

print(f'prompt: {prompt}')

prompt: <bos><|turn>system
你是一個繁體中文 AI 助理。
規則：
1. 可以由一般穩定知識回答的問題，直接回答。
2. 需要即時、近期或外部資訊時，使用適合的工具。
3. 如果有專門處理該資訊的工具，優先使用專用工具，不要使用較泛用的工具替代。
4. 不確定必要的工具參數時，先詢問使用者，不要自行猜測。
5. 工具回傳失敗時，不得捏造資料。
6. 最終回答使用繁體中文。
7. 面對具備多義性、地域差異或缺乏明確上下文的提問時，不得自行預設單一情境或立場。應客觀列出可能的多種情況，並主動引導使用者釐清確切意圖。<|tool>declaration:get_current_weather{description:<|"|>查詢指定地點目前的即時天氣。適用於目前溫度、體感溫度、濕度、降雨、雲量與風速等即時天氣資訊。如果使用者提供的是非英文地名，應將地名轉換成常用英文或羅馬拼音後再呼叫此工具。必須保留使用者真正指定的地點，不得自行替換成附近較大的城市。你需要對指定地點進行全世界可能所屬的國家判斷信心程度非常高可以明確推斷其所屬國家時，請自行推論並判斷該地的所屬國家，以代入工具的 country_code 參數當中。當信心程度有些微不確定無法肯定指定地點所屬的國家或第一行政區時，請先用自然語言回覆，表明不確定所詢問地點的所屬國家或其所屬行政區，並列出該地點所指的可能國家，讓使用者判斷回覆。<|"|>,parameters:{properties:{admin1:{description:<|"|>第一級行政區的英文名稱或正式縮寫。只有在使用者提供第一級行政區資訊時填入。例如：台北市 -> 'Taipei City'；California -> 'California'。此值將用來縮小 geocoding 搜尋結果。如果沒有提供或無法可靠判斷，可以省略。<|"|>,type:<|"|>STRING<|"|>},country_code:{description:<|"|>使用者提供國家資訊時，填入 ISO 3166-1 alpha-2 的兩碼大寫國家代碼。例如：台灣 -> 'TW'；日本 -> 'JP'；韓國 -> 'KR'；美國 -> 'US'。如果使用者沒有提供國家，且無法可靠判斷國家，可以省略此參數。<|"|>,type:<|"|>STRING

In [13]:
inputs = processor(
        text=prompt,
        return_tensors="pt",
    ).to(model.device)

print(f'inputs: {inputs}')

inputs: {'input_ids': tensor([[   2,  105, 9731,  ...,  105, 4368,  107]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), 'mm_token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0]], device='cuda:0')}


In [14]:
input_length = inputs["input_ids"].shape[-1]

outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        use_cache=True,
    )
generated_tokens = outputs[0][input_length:]

print(outputs)
print(generated_tokens)

tensor([[     2,    105,   9731,  ..., 236783,     49,     50]],
       device='cuda:0')
tensor([    48,   6639, 236787,    828, 236779,   4002, 236779,  19323, 236782,
          7125, 236787,     52, 236814,   5322,  36943,     52, 236764,  14801,
        236779,   3970, 236787,     52,  47747,     52, 236783,     49,     50],
       device='cuda:0')


In [15]:
processor.decode(
        generated_tokens,
        skip_special_tokens=False,
    )

'<|tool_call>call:get_current_weather{location:<|"|>Hsinchu<|"|>,country_code:<|"|>TW<|"|>}<tool_call|><|tool_response>'

In [16]:
import logging

logging.getLogger('transformers').setLevel(logging.ERROR)


@torch.inference_mode()
def generate_gemma(messages:list[dict[str,Any]], max_new_tokens:int=2048) -> str:
    """
    將 messages 與 tools 套用 gemma 4 chat template, 並取得模型原始輸出。
    """
    prompt = processor.apply_chat_template(
        messages,
        tools=TOOLS,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING
    )

    inputs = processor(
        text = prompt,
        return_tensors='pt'
    ).to(model.device)

    input_length = inputs['input_ids'].shape[-1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True
    )

    generated_tokens = outputs[0][input_length:]

    return processor.decode(
        generated_tokens,
        skip_special_tokens=False
    )


In [17]:
generated_output = generate_gemma(messages=messages)
print(generated_output)

<|tool_call>call:get_current_weather{location:<|"|>Hsinchu<|"|>,country_code:<|"|>TW<|"|>}<tool_call|><|tool_response>


In [18]:
from pydantic import ValidationError

def excute_tool_call(call:dict[str,Any]) -> dict[str,Any]:
    """
    驗證 function name, arguments 並執行允許的工具。
    """
    name = call.get('name')
    arguments = call.get('arguments',{})

    if name not in TOOL_REGISTRY:
        return {
            'name': name or 'unknown tool',
            'response': {
                'ok': False,
                'error': f'不允許或不存在的工具: {name}'
            }
        }

    tool = TOOL_REGISTRY[name]

    try:
        # Pydantic 參數驗證
        validated = tool['validator'].model_validate(arguments)

        # 執行 Python 函數
        result = tool['function'](**validated.model_dump())

        return {
            'name': name,
            'response':{
                'ok': True,
                'data':result
            }
        }
    except ValidationError as exc:
        return {
            'name': name,
            'response': {
                'ok': False,
                'error': '參數驗證未通過',
                'details': exc.errors(
                    include_url=False,
                    inclued_input=False
                )
            }
        }
    except requests.RequestException as exc:
        return {
            'name': name,
            'response': {
                'ok': False,
                'error': f'服務連線失敗:{exc}'
            }
        }
    except Exception as exc:
        return {
            'name': name,
            'response': {
                'ok': False,
                'error': str(exc)
            }
        }


In [19]:
parsed_output = parse_model_output(generated_output)
tool_calls = parsed_output['tool_calls']

print(parsed_output)
print(tool_calls)

for call in tool_calls:
    result = excute_tool_call(call)
    print(result)

{'content': '', 'tool_calls': [{'name': 'get_current_weather', 'arguments': {'location': 'Hsinchu', 'country_code': 'TW'}}], 'thinking': None}
[{'name': 'get_current_weather', 'arguments': {'location': 'Hsinchu', 'country_code': 'TW'}}]
{'name': 'get_current_weather', 'response': {'ok': True, 'data': {'requested_location': 'Hsinchu', 'requested_admin1': None, 'requested_country_code': 'TW', 'resolved_location': 'Hsinchu，Taiwan，Taiwan', 'resolved_country_code': 'TW', 'latitude': 24.80361, 'longitude': 120.96861, 'timezone': 'Asia/Taipei', 'observation_time': '2026-10-03T12:45', 'temperature': 31.3, 'temperature_unit': '°C', 'apparent_temperature': 35.6, 'relative_humidity_percent': 59, 'precipitation_mm': 0.1, 'cloud_cover_percent': 44, 'wind_speed_kmh': 15.0, 'weather_code': 51, 'weather_description': '輕微毛毛雨', 'data_source': 'Open-Meteo'}}}


In [20]:
import copy

def initial_messages() -> list[dict[str,Any]]:
    return [
        {
            'role': 'system',
            'content': SYSTEM_PROMPT
        }
    ]

def run_agent(
        user_message: str,
        conversation: list[dict[str,Any]] | None = None
        ) -> tuple[
            str,
            list[dict[str, Any]],
            list[dict[str, Any]]
            ]:
    """
    function calling orchestration

    Returns:
        answer:
            最終自然語言回答
        messages:
            包含 tool call 與 tool response 的完整 LLM 對話歷史
        trace:
            方便除錯與觀察的 agent 執行記錄
    """
    messages = copy.deepcopy(conversation) if conversation else initial_messages()

    messages.append(
        {
            'role': 'user',
            'content': user_message
        }
    )

    trace = []

    for round_index in range(MAX_TOOL_ROUNDS+1):
        raw_output = generate_gemma(messages = messages)
        parsed_output = parse_model_output(raw_output)
        calls = parsed_output['tool_calls']

        trace_item = {
            'round': round_index,
            'raw_model_output': raw_output,
            'parsed_tool_calls': calls
        }

        # 沒有 tool call，代表模型直接回答或已根據 tool result 進行回覆
        if not calls:
            answer = parsed_output['content'] or clean_generated_text(raw_output)

            if (
                messages[-1]['role'] == 'assistant'
                and 'tool_responses' in messages[-1]
                and 'content' not in messages[-1]
            ):
                messages[-1]['content'] = answer
            else:
                messages.append(
                    {
                        'role': 'assistant',
                        'content': answer
                    }
                )

            trace_item['final_answer'] = answer
            trace.append(trace_item)

            return answer, messages, trace

        # 模型要求執行工具
        tool_calls_for_history = []
        tool_responses = []

        for call in calls:
            tool_response = excute_tool_call(call)
            tool_responses.append(tool_response)

            tool_calls_for_history.append(
                {
                    'function':{
                        'name':call.get('name'),
                        'arguments': call.get('arguments',{})
                    }
                }
            )

        messages.append(
            {
            'role': 'assistant',
            'tool_calls': tool_calls_for_history,
            'tool_responses': tool_responses
                }
        )

        trace_item['tool_responses'] = tool_responses
        trace.append(trace_item)

    # 回到迴圈讓 generate_gemma 根據工具執行結果回答
    failure_message = '模型呼叫已達上限，無法完成這次請求。'

    messages.append(
        {
            'role': 'assistant',
            'content': failure_message
        }
    )

    return failure_message, messages, trace

In [21]:
answer, conversation, trace = run_agent('台北現在天氣如何?')

print('最終回答:')
print(answer)

最終回答:
台北目前的天氣是：

*   **溫度：** 28.9°C
*   **體感溫度：** 32.7°C
*   **天氣描述：** 輕微毛毛雨
*   **雲量：** 70%
*   **降雨量：** 0.1 mm
*   **相對濕度：** 70%
*   **風速：** 15.9 km/h

（資料來源：Open-Meteo，觀察時間：2026-10-03T12:45）


In [22]:
from pprint import pprint

pprint(trace)

[{'parsed_tool_calls': [{'arguments': {'country_code': 'TW',
                                       'location': 'Taipei'},
                         'name': 'get_current_weather'}],
  'raw_model_output': '<|tool_call>call:get_current_weather{location:<|"|>Taipei<|"|>,country_code:<|"|>TW<|"|>}<tool_call|><|tool_response>',
  'round': 0,
  'tool_responses': [{'name': 'get_current_weather',
                      'response': {'data': {'apparent_temperature': 32.7,
                                            'cloud_cover_percent': 70,
                                            'data_source': 'Open-Meteo',
                                            'latitude': 25.05306,
                                            'longitude': 121.52639,
                                            'observation_time': '2026-10-03T12:45',
                                            'precipitation_mm': 0.1,
                                            'relative_humidity_percent': 70,
                           

In [23]:
answer, _, trace = run_agent('請解釋 function calling 是什麼?')

print('最終回答:')
print(answer)

最終回答:
Function calling（函式呼叫）是一種在大型語言模型（LLM）中非常強大且實用的功能，它允許語言模型**理解用戶的意圖，並決定在需要時呼叫外部的、特定的程式碼或工具（即「函式」）來執行任務**。

簡單來說，它讓語言模型從一個「純粹的文本生成器」轉變為一個「能夠執行任務的智能代理（Agent）」。

以下我將從幾個面向來詳細解釋 Function Calling 的概念、工作原理和應用場景：

---

### 1. Function Calling 的核心概念

Function Calling 的流程可以概括為以下幾個步驟：

1. **定義工具（Define Tools/Functions）**：
   開發者會向語言模型提供一個或多個**外部函式的描述**。這個描述通常包含：
    * **函式的名稱 (Name)**：例如 `get_current_weather`。
    * **函式的用途描述 (Description)**：清楚說明這個函式是做什麼的。
    * **所需的參數 (Parameters)**：列出函式需要哪些輸入（例如：`location`、`country_code`）。
    * **參數的類型 (Type)**：說明每個參數的資料型態（例如：字串、數字）。

2. **用戶輸入 (User Input)**：
   用戶向模型提出一個請求，例如：「明天台北的天氣如何？」

3. **模型判斷與輸出 (Model Decision)**：
   語言模型接收到用戶的請求和它所知道的工具列表。模型會分析用戶的意圖，並判斷**是否需要使用其中一個工具來回答問題**。
    * **如果不需要**：模型會直接生成一個自然語言的回答。
    * **如果需要**：模型不會直接回答，而是會輸出一個**結構化的 JSON 格式的呼叫請求**，明確指出它想呼叫哪個函式以及需要傳入哪些參數。

4. **執行與回傳 (Execution and Return)**：
   系統（即應用程式的後端）會接收到模型輸出的結構化請求，**實際去執行對應的外部函式**（例如，呼叫 `get_current_weather` 工具）。執行完畢後，系統會將函式執行**的結果**（例如：溫度是 25°C，天氣晴朗）**再次傳回

In [24]:
answer, _, trace = run_agent('現在天氣如何？')

print(answer)
print(trace)

請問您想查詢哪個地點的天氣呢？請提供城市或地點名稱，以及國家（如果知道的話），我才能為您查詢。
[{'round': 0, 'raw_model_output': '請問您想查詢哪個地點的天氣呢？請提供城市或地點名稱，以及國家（如果知道的話），我才能為您查詢。<turn|>', 'parsed_tool_calls': [], 'final_answer': '請問您想查詢哪個地點的天氣呢？請提供城市或地點名稱，以及國家（如果知道的話），我才能為您查詢。'}]


In [25]:
conversation = None
SHOW_TOOL_TRACE = True

print("=" * 60)
print("Gemma 4 Tool Calling Agent")
print("=" * 60)
print()
print("指令：")
print("  exit     結束對話")
print("  clear    清除對話歷史")
print()

while True:
    user_message = input('\nYou:').strip()

    if user_message.lower() in {'exit', 'quit', '離開'}:
        print('離開對話')
        break

    if user_message.lower() == 'clear':
        conversation = None
        print('[System] 對話歷史已清除')
        continue

    if not user_message:
        continue
    try:
        answer, conversation, trace = run_agent(
            user_message = user_message,
            conversation = conversation
        )

        if SHOW_TOOL_TRACE:
            for item in trace:
                tool_calls = item.get('parsed_tool_calls', [])
                tool_responses = item.get('tool_responses', [])

                if tool_calls:
                    print('\n[Tool Call]')
                    for call in tool_calls:
                        print(f'function: {call['name']}')
                        print(f'arguments: {call['arguments']}')

                if tool_responses:
                    print('\n[Tool Response]')
                    for response in tool_responses:
                        print(response)

        print('\nGemma:')
        print(answer)
    except Exception as e:
        print('\n[Error]')
        print(f'{type(e).__name__}: {e}')


Gemma 4 Tool Calling Agent

指令：
  exit     結束對話
  clear    清除對話歷史


You:請問板橋的天氣

Gemma:
請問您想查詢的是哪個國家或地區的板橋的天氣呢？

因為「板橋」這個地名在不同的國家或地區可能存在，為了給您最準確的天氣資訊，請您提供國家或地區的資訊，例如「台灣」或「台北市」。

You:新北市板橋

[Tool Call]
function: get_current_weather
arguments: {'admin1': 'New Taipei City', 'country_code': 'TW', 'location': 'Banqiao'}

[Tool Response]
{'name': 'get_current_weather', 'response': {'ok': True, 'data': {'requested_location': 'Banqiao', 'requested_admin1': 'New Taipei City', 'requested_country_code': 'TW', 'resolved_location': 'Banqiao District，Taipei，Taiwan', 'resolved_country_code': 'TW', 'latitude': 25.01427, 'longitude': 121.46719, 'timezone': 'Asia/Taipei', 'observation_time': '2026-10-03T12:45', 'temperature': 30.2, 'temperature_unit': '°C', 'apparent_temperature': 33.8, 'relative_humidity_percent': 64, 'precipitation_mm': 0.2, 'cloud_cover_percent': 58, 'wind_speed_kmh': 17.3, 'weather_code': 53, 'weather_description': '中等毛毛雨', 'data_source': 'Open-Meteo'}}}

Gemma:
根據我查詢的資訊，**新北市板橋**（以